# 03 — Tool Schema Design — Practical Examples

**Covers**: weak vs strong schemas, parameter types, constraints, validation, named tools

In [ ]:
import os, json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'

try:
    from jsonschema import validate, ValidationError
    HAS_JSONSCHEMA = True
except ImportError:
    HAS_JSONSCHEMA = False
    print('Install jsonschema: pip install jsonschema')
print('✅ Setup complete')

## 1. Weak vs Strong Schema — See the Difference

In [ ]:
WEAK_TOOL = [{
    'type': 'function',
    'function': {
        'name': 'search',
        'description': 'Search for things',
        'parameters': {'type': 'object', 'properties': {'q': {'type': 'string'}}, 'required': ['q']}
    }
}]

STRONG_TOOL = [{
    'type': 'function',
    'function': {
        'name': 'web_search',
        'description': '''Search the web for current information.\nUSE when: real-time data, post-2023 events, prices, news.\nDO NOT use for: math, historical facts, creative tasks.\nTip: use specific targeted queries, not vague ones.''',
        'parameters': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'Specific search query. Example: "Python 3.13 new features list 2024"'
                },
                'num_results': {
                    'type': 'integer',
                    'description': 'Number of results (1-10). Default 5.',
                    'minimum': 1, 'maximum': 10, 'default': 5
                }
            },
            'required': ['query'],
            'additionalProperties': False
        }
    }
}]

test_prompt = 'What were the major AI releases in 2024?'

print('=== WEAK SCHEMA ===')
r1 = client.chat.completions.create(model=MODEL, messages=[{'role': 'user', 'content': test_prompt}], tools=WEAK_TOOL, tool_choice='required')
tc1 = r1.choices[0].message.tool_calls[0]
print(f'Name: {tc1.function.name}, Args: {tc1.function.arguments}')

print('\n=== STRONG SCHEMA ===')
r2 = client.chat.completions.create(model=MODEL, messages=[{'role': 'user', 'content': test_prompt}], tools=STRONG_TOOL, tool_choice='required')
tc2 = r2.choices[0].message.tool_calls[0]
print(f'Name: {tc2.function.name}, Args: {tc2.function.arguments}')

## 2. All JSON Schema Parameter Types

In [ ]:
COMPLEX_TOOL = [{
    'type': 'function',
    'function': {
        'name': 'create_report',
        'description': 'Create a structured report with all field types.',
        'parameters': {
            'type': 'object',
            'properties': {
                'title':        {'type': 'string', 'maxLength': 100, 'description': 'Report title'},
                'priority':     {'type': 'string', 'enum': ['low', 'medium', 'high', 'critical']},
                'score':        {'type': 'number', 'minimum': 0.0, 'maximum': 10.0},
                'page_count':   {'type': 'integer', 'minimum': 1},
                'is_public':    {'type': 'boolean'},
                'tags':         {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
                'metadata':     {
                    'type': 'object',
                    'properties': {
                        'author': {'type': 'string'},
                        'year':   {'type': 'integer'}
                    }
                }
            },
            'required': ['title', 'priority', 'score']
        }
    }
}]

r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Create a high-priority AI safety report with score 8.5, 3 pages, public, tags: [safety, alignment, research]'}],
    tools=COMPLEX_TOOL, tool_choice='required'
)
args = json.loads(r.choices[0].message.tool_calls[0].function.arguments)
print(json.dumps(args, indent=2))

## 3. Schema Validation

In [ ]:
def validate_tool_args(tool_name: str, args: dict, tools: list) -> tuple:
    tool = next((t for t in tools if t['function']['name'] == tool_name), None)
    if not tool:
        return False, f'Tool {tool_name} not found'
    schema = tool['function']['parameters']
    if HAS_JSONSCHEMA:
        try:
            validate(instance=args, schema=schema)
            return True, 'Valid'
        except ValidationError as e:
            return False, e.message
    return True, 'jsonschema not installed — skipping'

test_cases = [
    ('web_search', {'query': 'AI news', 'num_results': 5}),   # valid
    ('web_search', {'query': 'AI news', 'num_results': 50}),  # invalid: max is 10
    ('web_search', {}),                                        # invalid: missing required
]

for name, args in test_cases:
    ok, msg = validate_tool_args(name, args, STRONG_TOOL)
    status = '✅' if ok else '❌'
    print(f'{status} {name}({args}) → {msg}')